In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from scipy.stats import norm
from sklearn.linear_model import LinearRegression

SEASON = 2026
TARGET_WEEK = 7

PROJECT_ROOT = Path("../..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

ROS_OUTPUT_DIR = (
    PROJECT_ROOT
    / "rest_of_season"
    / "outputs"
    / f"week_{TARGET_WEEK:02d}"
)

ROS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
remaining_games = pd.read_parquet(
    ROS_OUTPUT_DIR / "remaining_schedule.parquet"
)

team_snapshot = pd.read_parquet(
    ROS_OUTPUT_DIR / "team_snapshot.parquet"
)

scoring_ratings = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_ros_scoring_ratings.parquet"
)

injury_strength = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

print("Remaining games:", len(remaining_games))
print("Teams:", len(team_snapshot))
print("Scoring ratings:", len(scoring_ratings))

assert len(team_snapshot) == 32
assert len(scoring_ratings) == 32
assert remaining_games["game_id"].is_unique

Remaining games: 179
Teams: 32
Scoring ratings: 32


In [3]:
strength_lookup = injury_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjusted_team_strength"
    ]
].copy()

assert strength_lookup["team"].is_unique
assert len(strength_lookup) == 32

home_strength = strength_lookup.rename(columns={
    "team": "home_team",
    "weekly_team_strength": "home_base_strength",
    "injury_adjusted_team_strength": "home_injury_strength"
})

away_strength = strength_lookup.rename(columns={
    "team": "away_team",
    "weekly_team_strength": "away_base_strength",
    "injury_adjusted_team_strength": "away_injury_strength"
})

games = (
    remaining_games
    .merge(
        home_strength,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_strength,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

games["home_model_strength"] = np.where(
    games["week"] == TARGET_WEEK,
    games["home_injury_strength"],
    games["home_base_strength"]
)

games["away_model_strength"] = np.where(
    games["week"] == TARGET_WEEK,
    games["away_injury_strength"],
    games["away_base_strength"]
)

games["neutral_strength_diff"] = (
    games["home_model_strength"]
    - games["away_model_strength"]
)

assert games[
    ["home_model_strength", "away_model_strength"]
].notna().all().all()

In [4]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = schedule_clean.filter(
    (pl.col("season") >= 2015)
    & (pl.col("season") <= 2025)
    & (pl.col("location") == "Home")
    & pl.col("result").is_not_null()
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()

rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

print("Home-field advantage:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest adjustment per day:", round(REST_POINT_VALUE, 3))
print("Historical margin SD:", round(historical_margin_std, 3))

Home-field advantage: 1.764
Rest adjustment per day: 0.181
Historical margin SD: 14.201


In [5]:
required_schedule_cols = [
    "location", "home_rest", "away_rest"
]

missing = [
    c for c in required_schedule_cols
    if c not in games.columns
]

if missing:
    raise KeyError(
        f"Remaining schedule missing columns: {missing}"
    )

if games[["home_rest", "away_rest"]].isna().any().any():
    raise ValueError(
        "Missing future rest-day values. "
        "We need to derive them from the schedule "
        "before generating projections."
    )

games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"] - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"] * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.5,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

In [6]:
home_scoring = scoring_ratings.rename(columns={
    "team": "home_team",
    "current_off_ppg": "home_current_off_ppg",
    "current_def_ppg": "home_current_def_ppg"
})

away_scoring = scoring_ratings.rename(columns={
    "team": "away_team",
    "current_off_ppg": "away_current_off_ppg",
    "current_def_ppg": "away_current_def_ppg"
})

games = (
    games
    .merge(
        home_scoring,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

scoring_cols = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[scoring_cols].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2

In [7]:
weekly_projection_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_projections.parquet"
)

weekly_games = pd.read_parquet(weekly_projection_path)

weekly_games = weekly_games.loc[
    weekly_games["week"] == TARGET_WEEK
].copy()

comparison = games.loc[
    games["week"] == TARGET_WEEK,
    [
        "game_id",
        "expected_home_margin",
        "projected_total",
        "home_win_probability"
    ]
].merge(
    weekly_games[
        [
            "game_id",
            "expected_home_margin",
            "projected_total",
            "home_win_probability"
        ]
    ],
    on="game_id",
    suffixes=("_ros", "_weekly"),
    validate="one_to_one"
)

assert len(comparison) == len(
    games.loc[games["week"] == TARGET_WEEK]
)

for metric in [
    "expected_home_margin",
    "projected_total",
    "home_win_probability"
]:
    assert np.allclose(
        comparison[f"{metric}_ros"],
        comparison[f"{metric}_weekly"],
        atol=1e-8
    ), f"Mismatch in {metric}"

print("Week 7 projections match the weekly model.")

Week 7 projections match the weekly model.


In [8]:
games["matchup"] = (
    games["away_team"] + " @ " + games["home_team"]
)

games["snapshot_week"] = TARGET_WEEK
games["season"] = SEASON

output_columns = [
    "season",
    "snapshot_week",
    "week",
    "game_id",
    "gameday",
    "away_team",
    "home_team",
    "matchup",
    "projected_away_score",
    "projected_home_score",
    "projected_total",
    "expected_home_margin",
    "away_win_probability",
    "home_win_probability",
    "predicted_winner",
    "predicted_win_probability"
]

ros_projections = games[
    output_columns
].sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert ros_projections["game_id"].is_unique
assert ros_projections["week"].ge(TARGET_WEEK).all()
assert ros_projections["week"].le(18).all()

ros_projections.to_parquet(
    ROS_OUTPUT_DIR / "ros_game_projections.parquet",
    index=False
)

ros_projections.to_csv(
    ROS_OUTPUT_DIR / "ros_game_projections.csv",
    index=False
)

print("Saved:", len(ros_projections), "remaining games")

display(ros_projections.head(20).round(3))

Saved: 179 remaining games


,season,snapshot_week,week,game_id,gameday,away_team,home_team,matchup,projected_away_score,projected_home_score,projected_total,expected_home_margin,away_win_probability,home_win_probability,predicted_winner,predicted_win_probability
0,2026,7,7,2026_07_NE_CHI,2026-10-22,NE,CHI,NE @ CHI,22.367,22.386,44.753,0.019,0.499,0.501,CHI,0.501
1,2026,7,7,2026_07_CIN_BAL,2026-10-25,CIN,BAL,CIN @ BAL,23.077,24.908,47.985,1.831,0.449,0.551,BAL,0.551
2,2026,7,7,2026_07_CLE_TEN,2026-10-25,CLE,TEN,CLE @ TEN,20.978,18.996,39.973,-1.982,0.555,0.445,CLE,0.555
3,2026,7,7,2026_07_DEN_ARI,2026-10-25,DEN,ARI,DEN @ ARI,26.136,19.154,45.290,-6.982,0.689,0.311,DEN,0.689
4,2026,7,7,2026_07_GB_DET,2026-10-25,GB,DET,GB @ DET,23.037,26.754,49.791,3.717,0.397,0.603,DET,0.603
5,2026,7,7,2026_07_IND_MIN,2026-10-25,IND,MIN,IND @ MIN,18.459,26.278,44.736,7.819,0.291,0.709,MIN,0.709
6,2026,7,7,2026_07_KC_SEA,2026-10-25,KC,SEA,KC @ SEA,21.561,23.481,45.042,1.920,0.446,0.554,SEA,0.554
7,2026,7,7,2026_07_LA_LV,2026-10-25,LA,LV,LA @ LV,25.826,21.979,47.805,-3.847,0.607,0.393,LA,0.607
8,2026,7,7,2026_07_MIA_NYJ,2026-10-25,MIA,NYJ,MIA @ NYJ,20.380,22.154,42.534,1.774,0.450,0.550,NYJ,0.550
9,2026,7,7,2026_07_NYG_HOU,2026-10-25,NYG,HOU,NYG @ HOU,17.832,27.561,45.392,9.729,0.247,0.753,HOU,0.753
